Used to analyze photometry data with TTL timestamps from MedAssociates software.
Curve fit and motion correction are based on Simpson et al. 2024 (https://doi.org/10.1016/j.neuron.2023.11.016). 
Cursor AI software was used to assist with writing parts of this code.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt  
import tdt
import pandas as pd
import os
from scipy.stats import linregress
from scipy.optimize import curve_fit
from collections import Counter

### Import Data


In [ ]:
# Assign each mouse to a group
group_id = {
    'mouseID':'groupname',
    'mouseID':'groupname'
    }

group_ids = list(set(group_id.values()))

In [ ]:
# Identifying session names and data locations for multiple sessions run on the same animals 
# List every session you want combined in this analysis.
SESSIONS = [
    {'session_name': 'Session 1', 'data_directory': './session1data'},
    {'session_name': 'Session 2', 'data_directory': './session2data'},
    {'session_name': 'Session 3', 'data_directory': './session3data'}
]

# Where to save Excel exports
data_output = './Output'

# Prefix for output filenames. If None, session names are joined with '_'.
output_basename = None
if output_basename is None:
    output_basename = '_'.join(s['session_name'] for s in SESSIONS)

# Extract metadata from folder names and read TDT blocks (1 or 2 animals per tank folder).
masterdat = []
for sess in SESSIONS:
    session_name = sess['session_name']
    data_directory = sess['data_directory']
    folderlist = [item for item in os.listdir(data_directory) if os.path.isdir(os.path.join(data_directory, item))]
    for foldername in folderlist:
        dat = {}
        dat['session_name'] = session_name

        if len(foldername) == 18:
            dat['mouse1'] = foldername[0:4]
            dat['mouse2'] = '0000'
            dat['date'] = foldername[5:11]
        else:
            dat['mouse1'] = foldername[0:4]
            dat['mouse2'] = foldername[5:9]
            dat['date'] = foldername[10:16]

        dat['blockpath'] = foldername
        dat['data'] = tdt.read_block(os.path.join(data_directory, dat['blockpath']))
        masterdat.append(dat)

### Pulling data and TTLs for each mouse

In [ ]:
#The "_470A" etc. values and epocs may need to be changed if recording from a different photometry setup

alldat = []

for f in range(len(masterdat)):
    
    if masterdat[f]['mouse1'] != '0000':
        dat1 = {}
        dat1['session_name'] = masterdat[f]['session_name']
        dat1['mouseID'] = masterdat[f]['mouse1']
        dat1['green'] = masterdat[f]['data'].streams._470A.data
        dat1['isos'] = masterdat[f]['data'].streams._405A.data
        dat1['TTL_values'] = masterdat[f]['data'].epocs.BOX1.data
        dat1['TTL_timestamps'] = masterdat[f]['data'].epocs.BOX1.onset
        dat1['sampling_rate'] = masterdat[f]['data'].streams._470A.fs
        alldat.append(dat1)
    
    if masterdat[f]['mouse2'] != '0000':
        dat2 = {}
        dat2['session_name'] = masterdat[f]['session_name']
        dat2['mouseID'] = masterdat[f]['mouse2']
        dat2['green'] = masterdat[f]['data'].streams._470B.data
        dat2['isos'] = masterdat[f]['data'].streams._405B.data
        dat2['TTL_values'] = masterdat[f]['data'].epocs.BOX2.data
        dat2['TTL_timestamps'] = masterdat[f]['data'].epocs.BOX2.onset 
        dat2['sampling_rate'] = masterdat[f]['data'].streams._470B.fs
        alldat.append(dat2)

# List mouse IDs and flag any mouse missing one or more configured sessions
from collections import defaultdict

expected_sessions = [s['session_name'] for s in SESSIONS]
sessions_by_mouse = defaultdict(set)
for entry in alldat:
    sessions_by_mouse[entry['mouseID']].add(entry['session_name'])

mouse_ids_sorted = sorted(sessions_by_mouse.keys())
print('Loaded mouse IDs:', mouse_ids_sorted)
print('Expected sessions:', expected_sessions)
for mid in mouse_ids_sorted:
    missing = [s for s in expected_sessions if s not in sessions_by_mouse[mid]]
    if missing:
        print(f'  FLAG: {mid} missing session(s): {missing}')
    else:
        print(f'  OK: {mid} - data for all {len(expected_sessions)} session(s)')

Additional preprocessing

In [ ]:
#assigning group name
for f in range(len(alldat)):
    alldat[f]['group'] = group_id[alldat[f]['mouseID']]
    alldat[f]['record_id'] = f"{alldat[f]['session_name']}_{alldat[f]['mouseID']}"

# cut so length matches
for f in range(len(alldat)):
    
    a = len(alldat[f]['green'])
    b = len(alldat[f]['isos'])
    if b < a:
        alldat[f]['green'] = alldat[f]['green'][0:b]
    if a < b:
        alldat[f]['isos'] = alldat[f]['isos'][0:a]

# Get time series in seconds
for f in range(len(alldat)):
    alldat[f]['time'] = (np.arange(1,len(alldat[f]['green'])+1))/alldat[f]['sampling_rate']

### Trim and and downsample

In [ ]:
# trim start
    
t = 8 # time threshold below which we will discard

for f in range(len(alldat)):
    indA = np.argmax(alldat[f]['time'] > t) # find first index of when time crosses threshold
    alldat[f]['time'] = alldat[f]['time'][indA:] # reformat vector to only include allowed time
    alldat[f]['trimmed_green'] = alldat[f]['green'][indA:]
    alldat[f]['trimmed_isos'] = alldat[f]['isos'][indA:]

In [ ]:
# Trim end

end_trim = 10  # Remove last n seconds from the end of each recording

for f in range(len(alldat)):
    # Get the total duration of the recording
    T = alldat[f]['time'][-1]
    
    # Define the new end threshold
    t_end = T - end_trim
    
    # Find the first index where time exceeds t_end
    indB = np.argmax(alldat[f]['time'] > t_end)
    
    # Edge case: if the last time is still below or equal to t_end,
    # it means there's effectively nothing to remove.
    if indB == 0 and alldat[f]['time'][-1] <= t_end:
        continue
    else:
        # Trim up to (not including) indB
        alldat[f]['time'] = alldat[f]['time'][:indB]
        alldat[f]['trimmed_green'] = alldat[f]['trimmed_green'][:indB]
        alldat[f]['trimmed_isos'] = alldat[f]['trimmed_isos'][:indB]

In [ ]:
# Downsample and average 

N = 100 # Average every N samples into 1 value

for f in range(len(alldat)):
    F_green=[]
    for i in range(0, len(alldat[f]['trimmed_green']), N):
        small_list = np.mean(alldat[f]['trimmed_green'][i:i+N])
        F_green.append(small_list)
    alldat[f]['downsampled_green'] = np.array(F_green)
    
    F_isos=[]
    for i in range(0, len(alldat[f]['trimmed_isos']), N):
        small_lst = np.mean(alldat[f]['trimmed_isos'][i:i+N])
        F_isos.append(small_lst)
    alldat[f]['downsampled_isos'] = np.array(F_isos)
    
    alldat[f]['downsampled_time'] = alldat[f]['time'][::N]
    

### Fit double exponential to each curve and use to correct for bleaching

In [ ]:
def double_exponential(t, const, amp_fast, amp_slow, tau_slow, tau_multiplier):
    """Compute a double exponential function with constant offset.
    Parameters
    ----------
    t : array-like
        Time vector in seconds
    const : float
        Amplitude of the constant offset
    amp_fast : float
        Amplitude of the fast component
    amp_slow : float
        Amplitude of the slow component
    tau_slow : float
        Time constant of slow component in seconds
    tau_multiplier : float
        Time constant of fast component relative to slow (tau_fast = tau_slow * tau_multiplier)
    
    Returns
    -------
    array-like
        The computed double exponential values at each time point
    """
    tau_fast = tau_slow * tau_multiplier
    return const + amp_slow * np.exp(-t/tau_slow) + amp_fast * np.exp(-t/tau_fast)

def get_bounds(trace, timecourse, tau_slow_min=0.0001, tau_slow_max=30):
    """Calculate parameter bounds for double exponential fitting.
    
    This function determines appropriate bounds for the parameters used in double exponential
    fitting based on the characteristics of the input signal and time vector.
    
    Parameters
    ----------
    trace : array-like
        The signal trace to be fit
    timecourse : array-like
        Time vector corresponding to the trace data points
    tau_slow_min : float, optional
        Minimum value for tau_slow as a fraction of total time duration (default: 0.0001)
    tau_slow_max : float, optional
        Maximum value for tau_slow as a fraction of total time duration (default: 30)
    
    Returns
    -------
    tuple of lists
        Two lists containing the lower and upper bounds respectively for:
        [amp_min, amp_min, amp_min, time_constant_min, offset_min],
        [amp_max, amp_max, amp_max, time_constant_max, offset_max]
        These bounds are used for fitting the double exponential function parameters.
    """
    signal = trace
    
    # Amplitude bounds
    amp_min = 0  # Assuming amplitude cannot be negative
    amp_max = 2 * np.max(signal)  # Allowing for some flexibility

    # Time constant bounds based on the duration of the experiment
    time_constant_min = tau_slow_min * (timecourse[-1] - timecourse[0])  # Minimum time constant
    time_constant_max = tau_slow_max * (timecourse[-1] - timecourse[0])  # Maximum time constant

    # Offset bounds
    offset_min = np.min(signal) if np.min(signal) < 0 else 0  # Adjust based on signal characteristics
    offset_max = np.max(signal)

    return (
        [amp_min, amp_min, amp_min, time_constant_min, offset_min],
        [amp_max, amp_max, amp_max, time_constant_max, offset_max]
    )

def process_photometry_signals(alldat):
    """Process photometry signals by fitting and removing exponential bleaching curves.
    
    For each entry in alldat, fits double exponential curves to both signals and
    subtracts them to correct for bleaching. Creates side-by-side plots showing
    raw signals with fits and detrended signals.
    
    Parameters
    ----------
    alldat : list of dict
        List of data dictionaries containing photometry signals
    
    Returns
    -------
    None
        Modifies alldat in place, adding:
        - 'expfitgreen', 'expfitisos': fitted curves
        - 'detrended_green', 'detrended_isos': bleaching-corrected signals
    """
    for f in range(len(alldat)):
        # Get signals
        green = alldat[f]['downsampled_green']
        isos = alldat[f]['downsampled_isos']
        time = alldat[f]['downsampled_time']
        
        # Fit green signal
        max_sig = np.max(green)
        inital_params = [max_sig/2, max_sig/4, max_sig/4, 3600, 0.1]
        bounds = get_bounds(green, time)
        green_parms, _ = curve_fit(double_exponential, time, green, 
                                 p0=inital_params, bounds=bounds, maxfev=1000)
        green_expfit = double_exponential(time, *green_parms)
        alldat[f]['expfitgreen'] = green_expfit
        
        # Fit isosbestic signal
        max_sig = np.max(isos)
        inital_params = [max_sig/2, max_sig/4, max_sig/4, 3600, 0.1]
        bounds = get_bounds(isos, time)
        isos_parms, _ = curve_fit(double_exponential, time, isos, 
                                 p0=inital_params, bounds=bounds, maxfev=1000)
        isos_expfit = double_exponential(time, *isos_parms)
        alldat[f]['expfitisos'] = isos_expfit
        
        # Detrend signals
        green_detrended = green - green_expfit
        isos_detrended = isos - isos_expfit
        alldat[f]['detrended_green'] = green_detrended
        alldat[f]['detrended_isos'] = isos_detrended

        # Create figure with two subplots side by side
        fig = plt.figure(figsize=(20, 8))
        
        # Plot 1: Raw signals with fits
        ax1 = plt.subplot(1, 2, 1)
        plot1 = ax1.plot(time, green, 'g', label='green')
        plot3 = ax1.plot(time, green_expfit, 'k', linewidth=1.5, label='Exponential fit')
        ax1.set_xlabel('Time (seconds)')
        ax1.set_ylabel('Green Signal (V)', color='g')
        ax1.tick_params(axis='y', labelcolor='g')
        
        ax1_twin = ax1.twinx()
        plot2 = ax1_twin.plot(time, isos, 'b', label='isos')
        plot4 = ax1_twin.plot(time, isos_expfit, 'k', linewidth=1.5)
        ax1_twin.set_ylabel('Isos Signal (V)', color='b')
        ax1_twin.tick_params(axis='y', labelcolor='b')
        
        # Combine legends
        lines = plot1 + plot2 + plot3
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper right')
        ax1.set_title(f"{alldat[f]['mouseID']} - Raw Signals")
        
        # Plot 2: Detrended signals
        ax2 = plt.subplot(1, 2, 2)
        plot5 = ax2.plot(time, green_detrended, 'g', label='green')
        ax2.set_xlabel('Time (seconds)')
        ax2.set_ylabel('Green Signal (V)', color='g')
        ax2.tick_params(axis='y', labelcolor='g')
        
        ax2_twin = ax2.twinx()
        plot6 = ax2_twin.plot(time, isos_detrended, 'b', label='isos')
        ax2_twin.set_ylabel('Isos Signal (V)', color='b')
        ax2_twin.tick_params(axis='y', labelcolor='b')
        
        # Combine legends
        lines = plot5 + plot6
        labels = [l.get_label() for l in lines]
        ax2.legend(lines, labels, loc='upper right')
        ax2.set_title(f"{alldat[f]['mouseID']} - Detrended Signals")
        
        plt.tight_layout()
        plt.show() 

def plot_motion_correction(time, isos_detrended, green_detrended, green_corrected, green_est_motion, slope, r_value, mouseID):
    """Plot motion correction analysis with correlation and correction plots side by side.
    
    Parameters
    ----------
    time : array-like
        Time points for x-axis
    isos_detrended : array-like
        Detrended isosbestic signal
    green_detrended : array-like
        Detrended green signal before motion correction
    green_corrected : array-like
        Motion-corrected green signal
    green_est_motion : array-like
        Estimated motion artifact
    slope : float
        Regression slope between isos and green
    r_value : float
        Correlation coefficient
    mouseID : str
        Mouse identifier for plot title
    """
    fig = plt.figure(figsize=(20, 8))
    
    # Plot 1: Correlation scatter plot
    ax1 = plt.subplot(1, 2, 1)
    ax1.scatter(isos_detrended[::5], green_detrended[::5], alpha=0.1, marker='.')
    x = np.array([np.min(isos_detrended), np.max(isos_detrended)])
    intercept = np.mean(green_detrended) - slope * np.mean(isos_detrended)
    ax1.plot(x, intercept + slope*x, 'b', linewidth=2)  # Changed from 'r' to 'b'
    ax1.set_xlabel('Isos')
    ax1.set_ylabel('Green')
    ax1.set_title(f"{mouseID} - Motion Correlation\nSlope: {slope:.3f}, R²: {r_value**2:.3f}")
    
    # Plot 2: Signal correction
    ax2 = plt.subplot(1, 2, 2)
    plot1 = ax2.plot(time, green_detrended, 'b', label='Pre motion correction', alpha=0.5)
    plot2 = ax2.plot(time, green_corrected, 'g', label='Motion corrected', alpha=0.5)
    plot3 = ax2.plot(time, green_est_motion - 1.05, 'y', label='Estimated motion')
    
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Green Signal (V)')
    ax2.set_title(f"{mouseID} - Motion Correction")
    ax2.legend(loc='upper right')
    ax2.set_xlim(60, 120)  # 60 sec window
    
    plt.tight_layout()
    plt.show()

def process_motion_correction(alldat):
    """Process motion correction for all recordings.
    
    Parameters
    ----------
    alldat : list of dict
        List of data dictionaries containing photometry signals
    """
    for f in range(len(alldat)):
        # Get detrended signals
        green_detrended = alldat[f]['detrended_green']
        isos_detrended = alldat[f]['detrended_isos']
        time = alldat[f]['downsampled_time']

        # Calculate correlation between signals
        slope, intercept, r_value, p_value, std_err = linregress(x=isos_detrended, y=green_detrended)
        print(f"\nMouse {alldat[f]['mouseID']}:")
        print('Slope    : {:.3f}'.format(slope))
        print('R-squared: {:.3f}'.format(r_value**2))

        # Calculate and remove motion effect
        green_est_motion = intercept + slope * isos_detrended
        green_corrected = green_detrended - green_est_motion
        alldat[f]['corrected_green'] = green_corrected

        # Plot correlation and correction
        plot_motion_correction(time, isos_detrended, green_detrended, 
                             green_corrected, green_est_motion, 
                             slope, r_value, alldat[f]['mouseID']) 

Curve fit

In [ ]:
# Process and plot photometry signals
process_photometry_signals(alldat)

Motion correction

In [ ]:
# Process motion correction
process_motion_correction(alldat)

### Extracting snips of corrected data

In [ ]:
#Set parameters for peri-event Z scoring

#Peri-event windows
PRE_TIME = 20 #  seconds before event to include
POST_TIME = 20 #  seconds after event to include

Base_start_trial = -20
Base_end_trial = -16

fs = alldat[0]['sampling_rate']/N; # must account for downsampling w/ N, assuming sampling rate is same in all mice

#time span for peri-event filtering        
TRANGE = [-1*PRE_TIME*np.floor(fs),POST_TIME*np.floor(fs)]

peri_time = range(int(TRANGE[1]-TRANGE[0]))/fs - PRE_TIME*np.floor(fs)/fs

In [ ]:
#Function to expand TTLs
def find_bits(sum_result):
    binary_representation = bin(sum_result)[2:][::-1]  # Reverse the binary string
    bits = [i for i, bit in enumerate(binary_representation) if bit == '1']
    return bits

In [ ]:
#Getting list of bit values and timestamps

for f in range(len(alldat)):

    bit_values=[]
    bit_timestamps=[]

    for TTL in range(len(alldat[f]['TTL_values'])):
        bits=find_bits(alldat[f]['TTL_values'][TTL].astype(int))
        ts=alldat[f]['TTL_timestamps'][TTL]

        for bit in range(len(bits)):
            bit_values=np.append(bit_values, bits[bit])
            bit_timestamps=np.append(bit_timestamps, ts)

    alldat[f]['bit_values']=bit_values
    alldat[f]['bit_timestamps']=bit_timestamps

In [ ]:
#Occasionally MedAssociates delivers duplicate TTLs within a short time window.
#Here we are screening for duplicate TTLs and excluding them

dt_max = 0.1  # 100 ms if timestamps are in seconds

for f in range(len(alldat)):
    bv = np.asarray(alldat[f]["bit_values"])
    bt = np.asarray(alldat[f]["bit_timestamps"])

    order = np.argsort(bt, kind="mergesort")
    bv = bv[order]
    bt = bt[order]

    mouse_label = alldat[f]["mouseID"]
    dup_elim = Counter()

    if len(bv) == 0:
        alldat[f]["bit_values_screened"] = bv.copy()
        alldat[f]["bit_timestamps_screened"] = bt.copy()
        print(f"{mouse_label}: no events; duplicates eliminated: (none)")
        continue

    last_t_for_bit = {}
    keep_idx = []

    for i in range(len(bv)):
        b = int(bv[i])
        t = bt[i]
        if b in last_t_for_bit and (t - last_t_for_bit[b]) < dt_max:
            dup_elim[b] += 1
            continue
        keep_idx.append(i)
        last_t_for_bit[b] = t

    keep_idx = np.array(keep_idx, dtype=int)
    alldat[f]["bit_values_screened"] = bv[keep_idx]
    alldat[f]["bit_timestamps_screened"] = bt[keep_idx]

    if not dup_elim:
        print(f"{mouse_label}: duplicates eliminated: none")
    else:
        parts = [f"bit {b}: {n}" for b, n in sorted(dup_elim.items())]
        total = sum(dup_elim.values())
        print(f"{mouse_label}: duplicates eliminated (total {total}) — " + "; ".join(parts))

In [ ]:
#Get snips of trace surrounding each timestamp

#this gives the number of datapoints that should be in each snip
correct_size = TRANGE[1]-TRANGE[0]

for f in range(len(alldat)): 

    Bit_list = sorted(set(alldat[f]['bit_values_screened'].astype(int)))

    Bit_snips = {}
    Bit_means = {}

    for Bit in Bit_list:

        Bitname = 'Bit_'+str(Bit)
        Beh_bits = np.where(alldat[f]['bit_values_screened'] == Bit)[0]
        Beh_t = alldat[f]['bit_timestamps_screened'][Beh_bits]
            
        trials = len(Beh_t)
        dFF_snips = [None] * trials
        array_ind = np.zeros(trials)
        pre_stim = np.zeros(trials)
        post_stim = np.zeros(trials)
            
        for i in range(trials):
                
            # Find first time index after bout onset
            array_ind[i] = np.argmax(alldat[f]['downsampled_time'] > Beh_t[i])

            # Find index corresponding to pre and post stim durations, making sure they're within the bounds of the trace
            pre_stim[i] = max(0, (array_ind[i] + TRANGE[0]))
            post_stim[i] = min((array_ind[i] + TRANGE[1]), len(alldat[f]['downsampled_time'])) 

            dFF_snips[i] = alldat[f]['corrected_green'][int(pre_stim[i]):int(post_stim[i])]
            

        #ignore any NANs or short snips caused by events too close to start or end of trace
        dFF_snips = [arr for arr in dFF_snips if isinstance(arr, np.ndarray) and not np.isnan(arr).any()]
        dFF_snips = [arr for arr in dFF_snips if len(arr) == correct_size]
            
        #Convert to a matrix. Store in 'Bit_snips' within each dat                
        Bit_snips[Bitname] = np.array(dFF_snips)

        #Get mean and stdev for each bit
        Bit_means[Bitname] = np.mean(np.array(dFF_snips), axis=0)


    alldat[f]['Bit_snips'] = Bit_snips #Call these by 'Bit_0' etc. 
    alldat[f]['Bit_means'] = Bit_means    

In [ ]:
#Plot bits, averaged per mouse - one subplot per session

bit_to_plot = 'Bit_4'

session_order = [s['session_name'] for s in SESSIONS]
n = len(session_order)
fig, axes = plt.subplots(n, 1, figsize=(9, max(4, 5 * n)), sharex=True, sharey=True)
if n == 1:
    axes = [axes]

for ax, sess in zip(axes, session_order):
    for f in range(len(alldat)):
        if alldat[f]['session_name'] != sess:
            continue
        if bit_to_plot in alldat[f]['Bit_means']:
            ax.plot(peri_time, alldat[f]['Bit_means'][bit_to_plot],
                    label=str(alldat[f]['mouseID'] + '_' + bit_to_plot))
    ax.set_title(f'Bit means - {sess}')
    ax.legend(loc='upper left')

plt.tight_layout()

In [ ]:
#Plot average (mean ± SEM across mice, split by group and session)

bit_to_plot = 'Bit_4'
session_order = [s['session_name'] for s in SESSIONS]
palette = plt.cm.tab20(np.linspace(0, 1, 20))
line_i = 0

for grp_id in group_ids:
    for sess in session_order:
        bit_arrays = []
        for f in range(len(alldat)):
            if alldat[f]['group'] == grp_id and alldat[f]['session_name'] == sess:
                if bit_to_plot in alldat[f]['Bit_means']:
                    bit_arrays.append(alldat[f]['Bit_means'][bit_to_plot])
        if not bit_arrays:
            continue
        bit_stack = np.vstack(bit_arrays)
        bit_mean = np.mean(bit_stack, axis=0)
        bit_sem = np.std(bit_stack, axis=0) / np.sqrt(bit_stack.shape[0])
        color = palette[line_i % len(palette)]
        line_i += 1
        plt.plot(peri_time, bit_mean, color=color, label=f'{grp_id} | {sess}')
        plt.fill_between(peri_time, bit_mean - bit_sem, bit_mean + bit_sem, alpha=0.3, color=color, label='_nolegend_')

plt.title('Bit means')
plt.legend()

Peri-event z scoring

In [ ]:
# calculating baseline time frame relative to window
Base_start_trial_calc = (Base_start_trial*np.floor(fs) - TRANGE[0]).astype(int) 
Base_end_trial_calc = (Base_end_trial*np.floor(fs) - TRANGE[0]).astype(int) 

for f in range(len(alldat)):  
        
    z_scores = {}
    z_means = {}
    z_sterr = {}

    Bit_list = sorted(set(alldat[f]['bit_values_screened'].astype(int))) 

    for Bit in Bit_list: 

        Bitname = 'Bit_'+str(Bit)

        trials = len(alldat[f]['Bit_snips'][Bitname])
        z_snips = [None] * trials

        for i in range(trials):
            zb = np.mean(alldat[f]['Bit_snips'][Bitname][i, Base_start_trial_calc:Base_end_trial_calc]) # baseline period mean 
            zsd = np.std(alldat[f]['Bit_snips'][Bitname][i, Base_start_trial_calc:Base_end_trial_calc]) # baseline period stdev
            z_snips[i]=(alldat[f]['Bit_snips'][Bitname][i,:] - zb)/zsd # Z score for each trial

        #Convert to a matrix. Store in 'Bit_snips' within each dat
        z_scores[Bitname] = np.array(z_snips)

        #Get mean and stdev for each bit        
        z_means[Bitname] = np.mean(np.array(z_snips), axis=0)
        z_sterr[Bitname] = np.std(np.array(z_snips), axis=0)/np.sqrt(np.array(z_snips).shape[0])
        
    alldat[f]['z_scores_trial'] = z_scores #Call these by 'Bit_0' etc. Trying to prevent mixups if not all bits are used
    alldat[f]['z_means_trial'] = z_means 
    alldat[f]['z_sterr_trial'] = z_sterr  


In [ ]:
#Plot peri event z scores - one subplot per session

bit_to_plot = 'Bit_4'

session_order = [s['session_name'] for s in SESSIONS]
n = len(session_order)
fig, axes = plt.subplots(n, 1, figsize=(9, max(4, 4.5 * n)), sharex=True, sharey=True)
if n == 1:
    axes = [axes]

for ax, sess in zip(axes, session_order):
    for f in range(len(alldat)):
        if alldat[f]['session_name'] != sess:
            continue
        if bit_to_plot in alldat[f]['z_means_trial']:
            ax.plot(peri_time, alldat[f]['z_means_trial'][bit_to_plot],
                    label=str(alldat[f]['mouseID'] + '_' + bit_to_plot))
    ax.set_title(f'Z means trial - {sess}')
    ax.legend(loc='upper left')

plt.tight_layout()

In [ ]:
#Plot average (mean ± SEM across mice, split by group and session)

bit_to_plot = 'Bit_4'
session_order = [s['session_name'] for s in SESSIONS]
palette = plt.cm.tab20(np.linspace(0, 1, 20))
line_i = 0

for grp_id in group_ids:
    for sess in session_order:
        bit_arrays = []
        for f in range(len(alldat)):
            if alldat[f]['group'] == grp_id and alldat[f]['session_name'] == sess:
                if bit_to_plot in alldat[f]['z_means_trial']:
                    bit_arrays.append(alldat[f]['z_means_trial'][bit_to_plot])
        if not bit_arrays:
            continue
        bit_stack = np.vstack(bit_arrays)
        bit_mean = np.mean(bit_stack, axis=0)
        bit_sterr = np.std(bit_stack, axis=0) / np.sqrt(bit_stack.shape[0])
        color = palette[line_i % len(palette)]
        line_i += 1
        plt.plot(peri_time, bit_mean, color=color, label=f'{grp_id} | {sess}')
        plt.fill_between(peri_time, bit_mean - bit_sterr, bit_mean + bit_sterr, alpha=0.3, color=color, label='_nolegend_')

plt.title('Z means_trial')
plt.legend()


Calculating AUC of z-scored signal per trial

In [ ]:
peri_time = np.asarray(peri_time, dtype=float)

auc_time_start = 0
auc_time_end = 10

mask = (peri_time >= auc_time_start) & (peri_time <= auc_time_end)
t_win = peri_time[mask]

if t_win.size < 2:
    raise ValueError("AUC window must include at least two time samples.")

# Store one AUC vector per session per bit
for f in range(len(alldat)):
    alldat[f]["z_trial_AUCs"] = {}
    ztrial = alldat[f]["z_scores_trial"]
    for Bitname, zmat in ztrial.items():
        zmat = np.asarray(zmat)
        # zmat: (n_trials, n_time); same n_time as peri_time
        z_win = zmat[:, mask]
        alldat[f]["z_trial_AUCs"][Bitname] = np.trapz(z_win, x=t_win, axis=1)

### Export files

In [ ]:
#Specify mice to export. Set to None to export all mice.
export_mouseIDs = ['mouse1', 'mouse2']  

In [ ]:
# Export means

export_column_order = 'session_group_mouse'  # could also sort by 'group' or 'mouse_id' first

# Specify which keys to export
export_keys = ['z_means_trial', 'Bit_means']  # Add or remove keys as needed

import re

def _mouse_id_sort_key(mid):
    """Sort key: numeric order for int/float and plain numeric strings; else first integer in string."""
    if isinstance(mid, (int, np.integer)):
        return (0, int(mid))
    if isinstance(mid, (float, np.floating)) and not (isinstance(mid, float) and np.isnan(mid)):
        return (0, int(mid))
    s = str(mid).strip()
    try:
        return (0, int(s))
    except ValueError:
        m = re.search(r'\d+', s)
        return (0, int(m.group(0))) if m else (1, s)

triallength = len(peri_time)

candidates = list(range(len(alldat)))
if export_mouseIDs is not None:
    allow = set(export_mouseIDs)
    candidates = [i for i in candidates if alldat[i]['mouseID'] in allow]

if export_column_order == 'session_group_mouse':
  sorted_indices = sorted(
      candidates,
      key=lambda i: (
          alldat[i]['session_name'],
          alldat[i]['group'],
          _mouse_id_sort_key(alldat[i]['mouseID']),
      ),
  )
elif export_column_order == 'group':
  sorted_indices = sorted(
      candidates,
      key=lambda i: (alldat[i]['group'], _mouse_id_sort_key(alldat[i]['mouseID'])),
  )
elif export_column_order == 'mouse_id':
  sorted_indices = sorted(candidates, key=lambda i: _mouse_id_sort_key(alldat[i]['mouseID']))
else:
  raise ValueError(
      "export_column_order must be 'session_group_mouse', 'group', or 'mouse_id'"
  )


colnames = [alldat[i]['mouseID'] for i in sorted_indices]
Bit_list = [0, 1, 2, 3, 4, 5, 6]

for key in export_keys:
    excel_file_path = os.path.join(data_output, f'{output_basename}_{key}.xlsx')
    os.makedirs(os.path.dirname(excel_file_path), exist_ok=True)

    with pd.ExcelWriter(excel_file_path, engine='xlsxwriter') as writer:
        for Bit in Bit_list:
            Bitname = 'Bit_' + str(Bit)
            zframe = np.empty((0, triallength))

            for i in sorted_indices:
                f = i
                if Bitname in alldat[f].get(key, {}):
                    data = alldat[f][key][Bitname]
                    if data is not None and np.any(~np.isnan(data)):
                        zframe = np.vstack((zframe, data))
                    else:
                        zframe = np.vstack((zframe, np.full((triallength,), np.nan)))
                else:
                    zframe = np.vstack((zframe, np.full((triallength,), np.nan)))

            df = pd.DataFrame(zframe).T
            df.columns = colnames
            df.insert(0, 'peri_time', peri_time)

            group_labels = [alldat[i]['group'] for i in sorted_indices]
            group_row = [''] + group_labels

            session_names = [alldat[i]['session_name'] for i in sorted_indices]
            session_row = [''] + session_names

            df_with_group = pd.concat(
                [pd.DataFrame([group_row], columns=df.columns), df],
                ignore_index=True,
            )

            df_with_session = pd.concat(
                [pd.DataFrame([session_row], columns=df.columns), df_with_group],
                ignore_index=True,
            )

            sheet_name = Bitname
            df_with_session.to_excel(writer, sheet_name=sheet_name, index=False)

In [ ]:
#Create 1 workbook per mouse that has bit_snips and zscores, for all trials for all bits

# Create subfolder path
subfolder_name = f"{output_basename}_individual_trials"
subfolder_path = os.path.join(data_output, subfolder_name)
os.makedirs(subfolder_path, exist_ok=True)

for f in range(len(alldat)):
    
    excel_file_path = os.path.join(subfolder_path, f"{alldat[f]['record_id']}_trials.xlsx")

    Bit_list = sorted(set(alldat[f]['bit_values_screened'].astype(int)))

    with pd.ExcelWriter(excel_file_path, engine='xlsxwriter') as writer:

        for Bit in Bit_list: 

            Bitname = 'Bit_'+str(Bit)  
            if Bitname in alldat[f]['Bit_snips']:
                Bit_snips = pd.DataFrame(alldat[f]['Bit_snips'][Bitname]).T
                name1 = str(Bitname +'_Bit_snips')
            if Bitname in alldat[f]['z_scores_trial']:
                Bit_zscore_trial = pd.DataFrame(alldat[f]['z_scores_trial'][Bitname]).T
                name2 = str(Bitname +'_zscores_trial')          

            Bit_snips.to_excel(writer, sheet_name=name1, index=False)
            Bit_zscore_trial.to_excel(writer, sheet_name=name2, index=False)

In [ ]:
#Exporting per trial AUCs

Bit_to_export = 4
Bitname = f'Bit_{Bit_to_export}'

# Explicit session order (sessions not listed sort after, alphabetically among extras)
session_order = [
    'Session 1',
    'Session 2',
    'Session 3'
]

output_path = os.path.join(data_output, f'{output_basename}_z_trial_AUCs_{Bitname}.xlsx')

allow = None if export_mouseIDs is None else set(export_mouseIDs)

rows = []
for f in range(len(alldat)):
    if allow is not None and alldat[f]['mouseID'] not in allow:
        continue
    auc_block = alldat[f].get('z_trial_AUCs', {})
    if Bitname not in auc_block:
        continue
    aucs = np.asarray(auc_block[Bitname], dtype=float).ravel()

    mid = alldat[f]['mouseID']
    sess = alldat[f]['session_name']
    for trial_idx, auc in enumerate(aucs):
        rows.append(
            {
                'mouseID': mid,
                'session': sess,
                'trial_within_session': trial_idx + 1,
                'z_trial_AUC': auc,
            }
        )

df = pd.DataFrame(rows)

# Sort: mouseID, then session by explicit list (unknown sessions last), then trial number
known = {s: i for i, s in enumerate(session_order)}
extra = sorted(set(df['session']) - set(session_order))
rank_map = {**known, **{s: len(known) + i for i, s in enumerate(extra)}}

df['_sess_rank'] = df['session'].map(rank_map)
df = df.sort_values(['mouseID', '_sess_rank', 'trial_within_session']).drop(columns=['_sess_rank'])
df = df.reset_index(drop=True)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_excel(output_path, index=False)